# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I built the feature vector from the repo-defined model features. The vector contains 18 numeric features and 8 categorical features. I used log1p versions of the main 90-day traffic counts because the traffic values are heavy-tailed. Numeric missing values are filled with 0 after cleaning, while categorical missing values are represented as unknown. I did not use content_id or client_id as features. The target is is_declining_label, which is 1 when trend_direction is down. I also excluded trend_direction and trend_pct because they are used to create the label.

In [5]:
# Section 1 — Build the feature vector

from pathlib import Path
import numpy as np
import pandas as pd

# Load the prepared feature vector if it already exists.
# Otherwise build it from the raw starter CSV.
ROOT = Path.cwd()

raw_path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
prepared_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

if prepared_path.exists():
    df = pd.read_csv(prepared_path)
    print(f"Loaded prepared data: {len(df):,} rows")
else:
    if not raw_path.exists():
        raise FileNotFoundError(
            "Could not find data/raw/content_refresh_anonymized.csv. "
            "Run the repo feature-preparation step first."
        )

    df = pd.read_csv(raw_path)

    # Numeric features used by the model.
    numeric_features = [
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
        "log_impressions_90d",
        "log_clicks_90d",
        "log_sessions_90d",
        "log_ai_sessions_90d",
        "days_with_impressions",
        "days_with_sessions",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
    ]

    categorical_features = [
        "competition_level",
        "content_type",
        "main_intent",
        "age_tier",
        "freshness_tier",
        "word_count_tier",
        "impression_tier",
        "position_tier",
    ]

    # Basic numeric conversion.
    for col in numeric_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Handle avg_position=0 as missing/no-position data.
    if "avg_position" in df.columns:
        df["avg_position"] = df["avg_position"].replace(0, np.nan)

    # Engineered log features for heavy-tailed traffic counts.
    df["log_impressions_90d"] = np.log1p(
        pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0)
    )
    df["log_clicks_90d"] = np.log1p(
        pd.to_numeric(df["clicks_90d"], errors="coerce").fillna(0)
    )
    df["log_sessions_90d"] = np.log1p(
        pd.to_numeric(df["sessions_90d"], errors="coerce").fillna(0)
    )
    df["log_ai_sessions_90d"] = np.log1p(
        pd.to_numeric(df["ai_sessions_90d"], errors="coerce").fillna(0)
    )

    # Categorical missing values.
    for col in categorical_features:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna("unknown")
                .astype(str)
                .replace({"": "unknown", "nan": "unknown"})
            )

    # Numeric missing values.
    for col in numeric_features:
        if col in df.columns:
            df[col] = (
                df[col]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0)
            )

    # Target.
    df["is_declining_label"] = (
        df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
    )

# Final feature lists.
MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

TARGET = "is_declining_label"

# Check that every requested feature exists.
feature_columns = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Missing model features: {missing_features}")

X = df[feature_columns].copy()
y = df[TARGET].copy()

print(f"Rows: {len(df):,}")
print(f"Numeric features: {len(MODEL_NUMERIC_FEATURES)}")
print(f"Categorical features: {len(MODEL_CATEGORICAL_FEATURES)}")
print(f"Total model features: {len(feature_columns)}")
print(f"Declining label rate: {y.mean():.3%}")

display(X.head())

Rows: 30,000
Numeric features: 18
Categorical features: 8
Total model features: 26
Declining label rate: 54.207%


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,scroll_rate,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,4.55,0.0,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,10.00,0.0,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,28.57,0.0,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,3.45,0.0,LOW,keyword article,commercial,365+,0-30,unknown,good,page_1
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,24.29,0.0,LOW,keyword article,informational,181-365,0-30,2000-3500,good,page_3_5


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
# Section 2 — Feature checks

print("Missing values after feature preparation:")
missing_summary = (
    X.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary.to_frame("missing_count"))

print("\nCategorical levels:")
for col in MODEL_CATEGORICAL_FEATURES:
    print(f"\n{col}:")
    print(X[col].value_counts(dropna=False).head(10))

print("\nData types:")
display(X.dtypes.to_frame("dtype"))

# Confirm the model feature matrix has no missing values.
assert not X.isna().any().any(), "Feature matrix still contains missing values."

print("PASS: feature matrix contains no missing values.")

Missing values after feature preparation:


,missing_count
search_volume,0
competition,0
cpc,0
word_count,0
char_count,0
log_impressions_90d,0
log_clicks_90d,0
log_sessions_90d,0
log_ai_sessions_90d,0
days_with_impressions,0



Categorical levels:

competition_level:
competition_level
LOW        22896
HIGH        2658
unknown     2610
MEDIUM      1836
Name: count, dtype: int64

content_type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

main_intent:
main_intent
informational    17235
transactional     5733
commercial        4612
unknown           2374
navigational        46
Name: count, dtype: int64

age_tier:
age_tier
91-180     11780
181-365    11368
365+        6360
31-90        492
Name: count, dtype: int64

freshness_tier:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64

word_count_tier:
word_count_tier
2000-3500    11263
unknown       7699
3500+         6285
1000-2000     3780
<1000          973
Name: count, dtype: int64

impression_tier:
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64

position_tier:

,dtype
search_volume,float64
competition,float64
cpc,float64
word_count,float64
char_count,float64
log_impressions_90d,float64
log_clicks_90d,float64
log_sessions_90d,float64
log_ai_sessions_90d,float64
days_with_impressions,int64


PASS: feature matrix contains no missing values.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature set for three leakage risks. First, trend_direction and trend_pct are label-derived and are not included. Second, the recent trend window must not overlap the outcome window; therefore I do not add trend_pct, trend_direction, or other outcome-window measurements as model inputs. Third, existing decision/product flags are excluded because they could reproduce an existing decision rather than learn an independent signal.

I also checked the final feature list programmatically for obvious label-related fields. The expected result is an empty list.

In [8]:
# Section 3 — Leakage hunt

# Fields that are explicitly known to be label-derived or unsafe.
known_leaky = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
}

# Existing decision/product-style fields that should not become model inputs.
decision_like_terms = [
    "decision",
    "flag",
    "score",
    "recommendation",
    "priority",
]

suspect_features = [
    col for col in feature_columns
    if col in known_leaky
    or any(term in col.lower() for term in decision_like_terms)
]

print("Suspect features found in model vector:")
print(suspect_features)

assert "is_declining_label" not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns

print("\nPASS: label-derived fields are excluded.")

# Show the relationship that creates the label.
if "trend_direction" in df.columns:
    derived_label = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

    print(
        f"Label consistency check: "
        f"{(derived_label == df[TARGET]).mean():.3%} rows agree."
    )

# Check that identifiers are not model features.
identifier_features = [
    col for col in ["content_id", "client_id"]
    if col in feature_columns
]

print("\nIdentifier fields in model features:", identifier_features)

assert not identifier_features
print("PASS: identifiers are not model features.")

# Direct leakage demonstration:
# trend_direction should perfectly determine the label because it defines the label.

if "trend_direction" in df.columns:
    leakage_table = pd.crosstab(
        df["trend_direction"],
        df[TARGET],
        margins=True
    )

    print("\nTrend direction vs target:")
    display(leakage_table)

Suspect features found in model vector:
[]

PASS: label-derived fields are excluded.
Label consistency check: 100.000% rows agree.

Identifier fields in model features: []
PASS: identifiers are not model features.

Trend direction vs target:


is_declining_label,0,1,All
trend_direction,,,
down,0,16262,16262
flat,1152,0,1152
new,2236,0,2236
stable,5962,0,5962
up,4388,0,4388
All,13738,16262,30000


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

content_id — pseudonymous identifier; useful for joins and checks, but not a predictive feature.

client_id — pseudonymous client identifier; reserved for grouped train/test splitting, not prediction.

trend_direction — directly used to create the target, so it is label leakage.

trend_pct — the numeric trend measure used to derive the target, so it is label leakage.

provider_used — content-generation metadata that the repo explicitly marks as not a model feature.

model_used — content-generation metadata that the repo explicitly marks as not a model feature.

is_declining_label — the target itself; including it as an input would directly reveal the answer.

has_clicks, has_ai_sessions, measurable_opportunity — preparation-derived fields that are not part of the repo-defined model feature list, so I did not add them without a modeling reason.

The goal is to keep the feature vector limited to information that could reasonably be available before the prediction moment.

In [9]:
# Section 4 — Explicit exclusion list

excluded_features = {
    "content_id": "Identifier only; used for joins/grouping, not prediction.",
    "client_id": "Identifier only; useful for grouped splits, not prediction.",
    "trend_direction": "Defines the target, so using it would leak the label.",
    "trend_pct": "Source of the target trend information; not safe as a feature.",
    "provider_used": "Generation metadata; repo does not define it as a model feature.",
    "model_used": "Generation metadata; repo does not define it as a model feature.",
    "is_declining_label": "The target itself; cannot be an input.",
    "has_clicks": "Preparation-derived field not included in the repo model feature list.",
    "has_ai_sessions": "Preparation-derived field not included in the repo model feature list.",
    "measurable_opportunity": "Preparation-derived field not included in the repo model feature list.",
}

print("Excluded fields:")
for field, reason in excluded_features.items():
    print(f"- {field}: {reason}")

# Verify that excluded fields are not accidentally present in X.
accidental = [
    field for field in excluded_features
    if field in X.columns
]

print("\nAccidentally included exclusions:", accidental)

assert not accidental
print("PASS: excluded fields are not in the feature matrix.")

Excluded fields:
- content_id: Identifier only; used for joins/grouping, not prediction.
- client_id: Identifier only; useful for grouped splits, not prediction.
- trend_direction: Defines the target, so using it would leak the label.
- trend_pct: Source of the target trend information; not safe as a feature.
- provider_used: Generation metadata; repo does not define it as a model feature.
- model_used: Generation metadata; repo does not define it as a model feature.
- is_declining_label: The target itself; cannot be an input.
- has_clicks: Preparation-derived field not included in the repo model feature list.
- has_ai_sessions: Preparation-derived field not included in the repo model feature list.
- measurable_opportunity: Preparation-derived field not included in the repo model feature list.

Accidentally included exclusions: []
PASS: excluded fields are not in the feature matrix.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.